# Alle Artikel klassifizieren mit NewsBERT-germ-210m

Laedt das fine-tuned Modell von HuggingFace, klassifiziert **alle** Artikel
(train + test + raw) und speichert das Ergebnis als CSV.

**Output-Spalten:**
- Alle Original-Spalten aus dem HuggingFace-Datensatz
- `split` — Herkunft: train / test / raw
- `prediction` — Klasse mit hoechstem Score
- `prediction_score` — Confidence der Top-Prediction
- `text_length_char` — Textlaenge in Zeichen
- `text_length_token` — Textlaenge in Tokens (EuroBERT Tokenizer)
- `score_{Klasse}` — Softmax-Wahrscheinlichkeit fuer jede der 13 Klassen

**Voraussetzung:** GPU-Runtime (L4 empfohlen), `HF_TOKEN` in Colab Secrets.

In [ ]:
# === SETUP ===
import os, sys

REPO = "/content/news_articles_classification_thesis"
if not os.path.exists(REPO):
    !git clone https://github.com/ZorbeyOezcan/news_articles_classification_thesis.git {REPO}
else:
    !cd {REPO} && git pull -q

!pip install -q transformers[sentencepiece] datasets huggingface_hub \
    scikit-learn matplotlib seaborn tqdm pandas accelerate

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PIPELINE_DIR = f"{REPO}/Python/classification_pipeline"
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

import importlib
import pipeline_utils as pu
importlib.reload(pu)

from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

print("Setup abgeschlossen.")

In [ ]:
# === KONFIGURATION ===
import torch
import numpy as np
import pandas as pd
from pathlib import Path

MODEL_REPO = "Zorryy/NewsBERT-germ-210m"
DATASET_ID = pu.DATASET_ID  # "Zorryy/news_articles_2025_elections_germany"
BATCH_SIZE = 16  # Inference batch size

ALL_LABELS = [
    "Klima / Energie", "Zuwanderung", "Renten", "Soziales Gef\u00e4lle",
    "AfD/Rechte", "Arbeitslosigkeit", "Wirtschaftslage", "Politikverdruss",
    "Gesundheitswesen, Pflege", "Kosten/L\u00f6hne/Preise",
    "Ukraine/Krieg/Russland", "Bundeswehr/Verteidigung", "Andere",
]

# Score-Spaltennamen (Sonderzeichen entfernt)
def safe_col_name(label):
    return ("score_" + label
            .replace(" / ", "_").replace("/", "_").replace(", ", "_")
            .replace(" ", "_")
            .replace("\u00e4", "ae").replace("\u00f6", "oe").replace("\u00fc", "ue"))

SCORE_COLUMNS = [safe_col_name(l) for l in ALL_LABELS]

# Output
OUTPUT_DIR = Path(pu.REPORTS_DIR)
OUTPUT_CSV = OUTPUT_DIR / "all_articles_classified.csv"

# GPU check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu_mem:.1f} GB)")
    if gpu_mem >= 20:
        BATCH_SIZE = 32
    elif gpu_mem >= 40:
        BATCH_SIZE = 64
else:
    print("WARNUNG: Keine GPU — Inference wird sehr langsam!")

print(f"Modell: {MODEL_REPO}")
print(f"Datensatz: {DATASET_ID}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Output: {OUTPUT_CSV}")

In [ ]:
# === DATENSATZ LADEN ===
from datasets import load_dataset

ds = load_dataset(DATASET_ID)
print(f"Splits: {list(ds.keys())}")
for split_name, split_ds in ds.items():
    print(f"  {split_name}: {len(split_ds):>7} Artikel")

# DataFrames mit split-Spalte erstellen
dfs = []
for split_name in ["train", "test", "raw"]:
    if split_name in ds:
        df = ds[split_name].to_pandas()
        df["split"] = split_name
        dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)
print(f"\nGesamt: {len(all_df)} Artikel")
print(f"Spalten: {list(all_df.columns)}")
print(f"\nSplit-Verteilung:")
print(all_df["split"].value_counts().to_string())

In [ ]:
# === MODELL & TOKENIZER LADEN ===
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers.modeling_rope_utils import ROPE_INIT_FUNCTIONS

# EuroBERT RoPE Fix
def _default_rope_init(config, device=None, **kwargs):
    base = getattr(config, "rope_theta", 10000.0)
    partial_rotary_factor = getattr(config, "partial_rotary_factor", 1.0)
    head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
    dim = int(head_dim * partial_rotary_factor)
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.int64).float().to(device) / dim))
    return inv_freq, 1.0

ROPE_INIT_FUNCTIONS["default"] = _default_rope_init

tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_REPO, trust_remote_code=True)
model = model.to(device)
model.eval()

# Label-Mapping aus Modell-Config
id2label = model.config.id2label
print(f"Modell geladen: {MODEL_REPO}")
print(f"Labels: {len(id2label)}")
for idx, label in sorted(id2label.items()):
    print(f"  {idx}: {label}")

In [ ]:
# === TEXT-LAENGEN BERECHNEN ===
from tqdm import tqdm

# Char-Laenge
all_df["text_length_char"] = all_df["text"].fillna("").str.len()

# Token-Laenge (EuroBERT Tokenizer, in Batches fuer Speichereffizienz)
print("Berechne Token-Laengen...")
texts = all_df["text"].fillna("").tolist()
token_lengths = []
TOKEN_BATCH = 1000

for i in tqdm(range(0, len(texts), TOKEN_BATCH), desc="Tokenisiere"):
    batch = texts[i:i + TOKEN_BATCH]
    encoded = tokenizer(batch, truncation=False, add_special_tokens=False)
    token_lengths.extend(len(ids) for ids in encoded["input_ids"])

all_df["text_length_token"] = token_lengths

print(f"\nChar-Laenge:  min={all_df['text_length_char'].min()}, "
      f"max={all_df['text_length_char'].max()}, "
      f"median={all_df['text_length_char'].median():.0f}")
print(f"Token-Laenge: min={all_df['text_length_token'].min()}, "
      f"max={all_df['text_length_token'].max()}, "
      f"median={all_df['text_length_token'].median():.0f}")

In [ ]:
# === BATCH-INFERENCE ===
from torch.utils.data import DataLoader, Dataset as TorchDataset

MAX_LENGTH = 2048

class TextDataset(TorchDataset):
    def __init__(self, texts, tokenizer, max_length):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )


def collate_fn(batch):
    return {
        "input_ids": torch.cat([b["input_ids"] for b in batch]),
        "attention_mask": torch.cat([b["attention_mask"] for b in batch]),
    }


texts = all_df["text"].fillna("").tolist()
dataset = TextDataset(texts, tokenizer, MAX_LENGTH)
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)

all_predictions = []
all_scores = []

print(f"Klassifiziere {len(texts)} Artikel (Batch Size={BATCH_SIZE})...")

with torch.no_grad():
    for batch in tqdm(dataloader, desc="Inference"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()

        pred_ids = probs.argmax(axis=-1)
        pred_labels = [id2label[int(i)] for i in pred_ids]

        all_predictions.extend(pred_labels)
        all_scores.extend(probs.tolist())

print(f"\nInference abgeschlossen: {len(all_predictions)} Predictions")

In [ ]:
# === ERGEBNISSE ZUSAMMENFUEGEN ===

# Prediction + Top-Score
all_df["prediction"] = all_predictions
scores_array = np.array(all_scores)
all_df["prediction_score"] = scores_array.max(axis=1)

# Score-Spalten fuer alle 13 Klassen
for i, col_name in enumerate(SCORE_COLUMNS):
    all_df[col_name] = scores_array[:, i]

# Spaltenreihenfolge festlegen
original_cols = ["id", "domain", "url", "date_time", "headline", "author",
                 "text", "text_length", "label", "split"]
new_cols = ["prediction", "prediction_score", "text_length_char", "text_length_token"]
final_cols = original_cols + new_cols + SCORE_COLUMNS

# Nur Spalten verwenden die tatsaechlich existieren
final_cols = [c for c in final_cols if c in all_df.columns]
all_df = all_df[final_cols]

print(f"Finale Tabelle: {all_df.shape[0]} Zeilen x {all_df.shape[1]} Spalten")
print(f"\nSpalten:")
for i, col in enumerate(all_df.columns):
    print(f"  {i+1:>2}. {col}")

print(f"\nVorschau (erste 3 Zeilen):")
display_cols = ["id", "headline", "label", "split", "prediction",
                "prediction_score", "text_length_char", "text_length_token"]
print(all_df[display_cols].head(3).to_string(index=False))

In [ ]:
# === CSV SPEICHERN ===
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
all_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

file_size_mb = OUTPUT_CSV.stat().st_size / 1e6
print(f"CSV gespeichert: {OUTPUT_CSV}")
print(f"Groesse: {file_size_mb:.1f} MB")
print(f"Zeilen: {len(all_df):,}")
print(f"Spalten: {len(all_df.columns)}")

In [ ]:
# === SUMMARY STATS ===
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("=" * 70)
print("  CLASSIFICATION SUMMARY")
print("=" * 70)

# --- 1. Artikel pro Split ---
print(f"\n1. Artikel pro Split:")
for split_name in ["train", "test", "raw"]:
    n = len(all_df[all_df["split"] == split_name])
    print(f"   {split_name:<6} {n:>7,}")
print(f"   {'TOTAL':<6} {len(all_df):>7,}")

# --- 2. Prediction-Verteilung (alle Artikel) ---
print(f"\n2. Prediction-Verteilung (alle {len(all_df):,} Artikel):")
pred_dist = all_df["prediction"].value_counts().sort_values(ascending=False)
for label, count in pred_dist.items():
    pct = count / len(all_df) * 100
    print(f"   {label:<30} {count:>7,} ({pct:>5.1f}%)")

# --- 3. Prediction-Verteilung (nur raw) ---
raw_df = all_df[all_df["split"] == "raw"]
print(f"\n3. Prediction-Verteilung (nur raw, {len(raw_df):,} Artikel):")
raw_pred_dist = raw_df["prediction"].value_counts().sort_values(ascending=False)
for label, count in raw_pred_dist.items():
    pct = count / len(raw_df) * 100
    print(f"   {label:<30} {count:>7,} ({pct:>5.1f}%)")

# --- 4. Uebereinstimmung Prediction vs. Label (train + test) ---
labeled_df = all_df[all_df["split"].isin(["train", "test"])].copy()
labeled_df = labeled_df[labeled_df["label"].notna() & (labeled_df["label"] != "")]

if len(labeled_df) > 0:
    acc = accuracy_score(labeled_df["label"], labeled_df["prediction"])
    f1_mac = f1_score(labeled_df["label"], labeled_df["prediction"],
                      average="macro", zero_division=0)
    f1_wt = f1_score(labeled_df["label"], labeled_df["prediction"],
                     average="weighted", zero_division=0)

    print(f"\n4. Uebereinstimmung Prediction vs. Label (train+test, n={len(labeled_df):,}):")
    print(f"   Accuracy:    {acc:.4f}")
    print(f"   F1 Macro:    {f1_mac:.4f}")
    print(f"   F1 Weighted: {f1_wt:.4f}")

    # Pro Split
    for split_name in ["train", "test"]:
        split_df = labeled_df[labeled_df["split"] == split_name]
        if len(split_df) > 0:
            s_acc = accuracy_score(split_df["label"], split_df["prediction"])
            s_f1 = f1_score(split_df["label"], split_df["prediction"],
                            average="macro", zero_division=0)
            print(f"   {split_name:<6} Accuracy={s_acc:.4f}, F1 Macro={s_f1:.4f} (n={len(split_df)})")

# --- 5. Confidence-Statistiken ---
print(f"\n5. Confidence-Statistiken (prediction_score):")
for split_name in ["train", "test", "raw"]:
    split_df = all_df[all_df["split"] == split_name]
    scores = split_df["prediction_score"]
    print(f"   {split_name:<6} mean={scores.mean():.4f}, "
          f"median={scores.median():.4f}, "
          f"min={scores.min():.4f}, "
          f"<0.5={int((scores < 0.5).sum()):,} ({(scores < 0.5).mean()*100:.1f}%)")

# --- 6. Text-Laengen ---
print(f"\n6. Text-Laengen:")
print(f"   Chars:  mean={all_df['text_length_char'].mean():.0f}, "
      f"median={all_df['text_length_char'].median():.0f}, "
      f"max={all_df['text_length_char'].max():,}")
print(f"   Tokens: mean={all_df['text_length_token'].mean():.0f}, "
      f"median={all_df['text_length_token'].median():.0f}, "
      f"max={all_df['text_length_token'].max():,}")

print(f"\n{'='*70}")
print(f"CSV: {OUTPUT_CSV}")
print(f"{'='*70}")

In [ ]:
# === PER-CLASS REPORT (train+test) ===
if len(labeled_df) > 0:
    print("Per-Class Classification Report (train+test vs. prediction):\n")
    print(classification_report(
        labeled_df["label"], labeled_df["prediction"],
        labels=ALL_LABELS, target_names=ALL_LABELS, zero_division=0,
    ))

In [ ]:
# === CLEANUP ===
import gc

del model, dataloader, dataset, scores_array, all_scores, all_predictions
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print(f"Fertig. CSV mit {len(all_df):,} Artikeln gespeichert.")
print(f"  {OUTPUT_CSV}")